# EDA для Feedback Assistant (ДЗ2)

Разбираем данные для проекта: помощник, который пишет черновик обратной связи по работам студентов.

Зачем этот ноутбук: понять, сколько у нас данных, нормального ли они качества и чего не хватает. Бизнес-постановку не повторяю — она уже в `docs/ml_system_design_doc.md`.

Откуда данные: кусок из командного репозитория `task-reviewer` (папки с архивом, тетрадями и PDF). Коротко про файлы написано в `data/README.md`.


## 0. Подготовка

In [1]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.titlesize"] = 12

ROOT = Path("..").resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = Path(".").resolve()

DATA = ROOT / "data" / "raw"
FIG = ROOT / "notebooks" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

print("DATA =", DATA)
print("exists:", DATA.exists())

DATA = /Users/timelysoftwork/Desktop/feedback/Feedback_Assistant/data/raw
exists: True


## 1. Состав и объём данных

Сначала просто смотрим, что вообще лежит в папках: задания, рубрики, работы, PDF.


In [2]:
archive = DATA / "archive"
ru_dir = DATA / "ru_feedback"
pdf_dir = DATA / "pdf"
pdf_text_dir = DATA / "pdf_text"

tasks = [
    json.loads(line)
    for line in (archive / "tasks_parsed.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]
checks_scored = [
    json.loads(line)
    for line in (archive / "checks_scored.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]
checks_parsed = [
    json.loads(line)
    for line in (archive / "checks_parsed.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

ru_files = sorted(ru_dir.glob("*.md"))
pdf_files = sorted(pdf_dir.glob("*.pdf"))
pdf_text_files = sorted(pdf_text_dir.glob("*.txt"))

inventory = pd.DataFrame(
    [
        {
            "сущность": "Задания + рубрики",
            "путь": "data/raw/archive/tasks_parsed.jsonl",
            "формат": "jsonl",
            "объём": len(tasks),
        },
        {
            "сущность": "Размеченные работы (scored)",
            "путь": "data/raw/archive/checks_scored.jsonl",
            "формат": "jsonl",
            "объём": len(checks_scored),
        },
        {
            "сущность": "Работы с исходной разметкой (parsed)",
            "путь": "data/raw/archive/checks_parsed.jsonl",
            "формат": "jsonl",
            "объём": len(checks_parsed),
        },
        {
            "сущность": "Русские тексты (тетради)",
            "путь": "data/raw/ru_feedback/",
            "формат": "md",
            "объём": len(ru_files),
        },
        {
            "сущность": "PDF работ / метрика",
            "путь": "data/raw/pdf/",
            "формат": "pdf",
            "объём": len(pdf_files),
        },
        {
            "сущность": "Извлечённый текст PDF",
            "путь": "data/raw/pdf_text/",
            "формат": "txt",
            "объём": len(pdf_text_files),
        },
    ]
)
inventory

,сущность,путь,формат,объём
0,Задания + рубрики,data/raw/archive/tasks_parsed.jsonl,jsonl,2
1,Размеченные работы (scored),data/raw/archive/checks_scored.jsonl,jsonl,8
2,Работы с исходной разметкой (parsed),data/raw/archive/checks_parsed.jsonl,jsonl,8
3,Русские тексты (тетради),data/raw/ru_feedback/,md,27
4,PDF работ / метрика,data/raw/pdf/,pdf,6
5,Извлечённый текст PDF,data/raw/pdf_text/,txt,6


In [3]:
print("Задания:")
for t in tasks:
    print(
        f"- {t['id']}: критериев = {len(t['criteria'])}, "
        f"длина формулировки = {len(t['task'].split())} слов"
    )

print("\nКритерии первого задания:")
for name, weight in tasks[0]["criteria"]:
    print(f"  [{weight}] {name}")

Задания:
- science_essay_250: критериев = 10, длина формулировки = 51 слов
- book_review_150_180: критериев = 10, длина формулировки = 56 слов

Критерии первого задания:
  [1] The text is relevant to the task and of the required size
  [1] The text engages the reader
  [1] The format and register are relevant to the task
  [1] Each point is sufficiently developed
  [1] The text is organised into logical paragraphs
  [1] Uses linkers, transition/introductory words covered in the course
  [1] Uses some new grammar structures fitting the task
  [1] Accurately uses grammar from the previous level
  [1] Uses some new vocabulary studied fitting the task
  [1] Makes no more than 3 errors in familiar vocabulary (spelling, collocations, meaning)


**Вывод по составу**

Для бейзлайна набор собран нормально: есть задание, рубрика примерно из 10 критериев и 8 английских работ с баллами и фидбэком.

Для русского MVP файлов больше — 27 текстов и несколько PDF. Но баллов по критериям у них нет. Размеченных работ мало: на обучение это слабо, на первый разбор и проверку идей — нормально.


## 2. Качество размеченного набора (английские эссе)

Смотрим длины ответов, нет ли пустых фидбэков и как распределены оценки по критериям.


In [4]:
rows = []
for item in checks_scored:
    answer = item.get("answer", "")
    scores = item.get("criteria_results") or item.get("criteria_result") or []
    rows.append(
        {
            "task_id": item["task_id"],
            "n_words": len(answer.split()),
            "n_chars": len(answer),
            "n_criteria": len(scores),
            "n_positive": sum(int(s) for _, s in scores),
            "has_feedback": bool(str(item.get("feedback", "")).strip()),
            "feedback_words": len(str(item.get("feedback", "")).split()),
        }
    )

scored_df = pd.DataFrame(rows)
scored_df

,task_id,n_words,n_chars,n_criteria,n_positive,has_feedback,feedback_words
0,science_essay_250,274,1803,10,7,True,387
1,science_essay_250,253,2009,10,9,True,405
2,science_essay_250,191,1287,10,5,True,379
3,book_review_150_180,135,728,10,6,True,204
4,book_review_150_180,190,1073,10,9,True,260
5,book_review_150_180,45,261,10,1,True,282
6,book_review_150_180,131,712,10,5,True,240
7,book_review_150_180,166,932,10,10,True,262


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(scored_df["n_words"], bins=8, ax=axes[0], color="#4C78A8")
axes[0].set_title("Длина ответа студента (слова)")
axes[0].set_xlabel("слов в answer")

sns.barplot(
    data=scored_df, x="task_id", y="n_words", ax=axes[1], color="#F58518", errorbar=None
)
axes[1].set_title("Длина ответа по заданию")
axes[1].tick_params(axis="x", rotation=20)
axes[1].set_xlabel("")
plt.tight_layout()
fig.savefig(FIG / "scored_answer_lengths.png", dpi=140, bbox_inches="tight")
plt.show()

print(scored_df[["n_words", "n_positive", "feedback_words"]].describe().round(1))
print("\nПустые feedback:", int((~scored_df["has_feedback"]).sum()))
print("Работ по заданиям:\n", scored_df["task_id"].value_counts().to_string())

       n_words  n_positive  feedback_words
count      8.0         8.0             8.0
mean     173.1         6.5           302.4
std       72.6         2.9            76.5
min       45.0         1.0           204.0
25%      134.0         5.0           255.0
50%      178.0         6.5           272.0
75%      206.5         9.0           381.0
max      274.0        10.0           405.0

Пустые feedback: 0
Работ по заданиям:
 task_id
book_review_150_180    5
science_essay_250      3


/var/folders/dp/1h0qmjhd05qc9y9v11b3h_vm0000gt/T/ipykernel_9283/3768740314.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Вывод**

Длины очень разные: где-то около 45 слов, где-то около 250. Пустых фидбэков нет. Но работ всего 8 на 2 задания, так что любые метрики на этой выборке будут шуметь.


In [6]:
crit_rows = []
for item in checks_scored:
    scores = item.get("criteria_results") or item.get("criteria_result") or []
    for name, score in scores:
        crit_rows.append(
            {"task_id": item["task_id"], "criterion": name, "score": int(score)}
        )

crit_df = pd.DataFrame(crit_rows)
balance = (
    crit_df.groupby("criterion")["score"].agg(["mean", "sum", "count"]).reset_index()
)
balance = balance.sort_values("mean")
balance["negative"] = balance["count"] - balance["sum"]
balance

,criterion,mean,sum,count,negative
6,The text is relevant to the task and of the re...,0.250,2,8,6
0,Accurately uses grammar from the previous level,0.375,3,8,5
1,Each point is sufficiently developed,0.375,3,8,5
4,The text engages the reader,0.375,3,8,5
5,The text is organised into logical paragraphs,0.750,6,8,2
2,Makes no more than 3 errors in familiar vocabu...,0.875,7,8,1
3,The format and register are relevant to the task,0.875,7,8,1
7,"Uses linkers, transition/introductory words co...",0.875,7,8,1
8,Uses some new grammar structures fitting the task,0.875,7,8,1
9,Uses some new vocabulary studied fitting the task,0.875,7,8,1


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = balance.sort_values("mean")
ax.barh(plot_df["criterion"], plot_df["mean"], color="#54A24B")
ax.set_xlim(0, 1)
ax.set_xlabel("доля выполненных (score=1)")
ax.set_title("Баланс меток по критериям (checks_scored)")
plt.tight_layout()
fig.savefig(FIG / "criteria_positive_rate.png", dpi=140, bbox_inches="tight")
plt.show()

print("Всего меток:", len(crit_df))
print(
    "score=1:",
    int((crit_df["score"] == 1).sum()),
    "score=0:",
    int((crit_df["score"] == 0).sum()),
)
print("Критерии почти всегда выполнены (mean >= 0.85):")
print(
    balance.loc[balance["mean"] >= 0.85, ["criterion", "mean"]].to_string(index=False)
)

Всего меток: 80
score=1: 52 score=0: 28
Критерии почти всегда выполнены (mean >= 0.85):
                                                                           criterion  mean
Makes no more than 3 errors in familiar vocabulary (spelling, collocations, meaning) 0.875
                                    The format and register are relevant to the task 0.875
                   Uses linkers, transition/introductory words covered in the course 0.875
                                   Uses some new grammar structures fitting the task 0.875
                                   Uses some new vocabulary studied fitting the task 0.875


/var/folders/dp/1h0qmjhd05qc9y9v11b3h_vm0000gt/T/ipykernel_9283/4149686203.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Вывод по разметке**

Оценки стоят как 0 или 1. Есть и плохие, и хорошие примеры, но по части критериев почти всегда стоит 1. Из-за этого F1 может выглядеть красиво просто потому, что модель чаще говорит «выполнено». Для бейзлайна набор ещё можно использовать. Для нормального обучения нужно больше работ и более ровная разметка.


In [8]:
parsed_map = {}
for item in checks_parsed:
    key = (item["task_id"], item["answer"][:80])
    parsed_map[key] = {name: int(s) for name, s in item.get("criteria_results", [])}

diffs = []
for item in checks_scored:
    key = (item["task_id"], item["answer"][:80])
    scored = {name: int(s) for name, s in (item.get("criteria_results") or [])}
    parsed = parsed_map.get(key)
    if not parsed:
        continue
    for name in scored:
        if name in parsed and parsed[name] != scored[name]:
            diffs.append(
                {
                    "task_id": item["task_id"],
                    "criterion": name,
                    "parsed": parsed[name],
                    "scored": scored[name],
                }
            )

diff_df = pd.DataFrame(diffs)
matched = sum(
    1 for item in checks_scored if (item["task_id"], item["answer"][:80]) in parsed_map
)
print("Пар работ matched:", matched)
print("Расхождений parsed vs scored:", len(diff_df))
diff_df.head(10)

Пар работ matched: 8
Расхождений parsed vs scored: 22


,task_id,criterion,parsed,scored
0,science_essay_250,The text engages the reader,1,0
1,science_essay_250,Each point is sufficiently developed,1,0
2,science_essay_250,Accurately uses grammar from the previous level,1,0
3,science_essay_250,Makes no more than 3 errors in familiar vocabu...,0,1
4,science_essay_250,The text is relevant to the task and of the re...,1,0
5,science_essay_250,The text engages the reader,1,0
6,science_essay_250,Each point is sufficiently developed,1,0
7,science_essay_250,The text is organised into logical paragraphs,0,1
8,science_essay_250,Accurately uses grammar from the previous level,1,0
9,book_review_150_180,The text engages the reader,1,0


**Вывод**

В `checks_parsed` и `checks_scored` баллы местами не совпадают. Чтобы не путаться, дальше за эталон берём `checks_scored.jsonl`.


## 3. Русский корпус (`Тетрадь_*.md`)

Похоже, это уже готовые разборы и черновики комментариев, а не сырые эссе с оценками. Смотрим длину, язык и нет ли в тексте почты или телефона.


In [9]:
def load_ru_text(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if "output_preview" in text[:100]:
        match = re.search(r'"output_preview"\s*:\s*"(.*)"\s*$', text, flags=re.S)
        if match:
            raw = match.group(1)
            try:
                return bytes(raw, "utf-8").decode("unicode_escape")
            except Exception:
                return raw.replace("\\n", "\n").replace('\\"', '"')
    return text


ru_rows = []
email_re = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
phone_re = re.compile(
    r"(?:\+7|8)[\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}"
)
cyr_re = re.compile(r"[а-яА-ЯёЁ]")
lat_re = re.compile(r"[A-Za-z]")

for path in ru_files:
    text = load_ru_text(path)
    cyr = len(cyr_re.findall(text))
    lat = len(lat_re.findall(text))
    ru_rows.append(
        {
            "file": path.name,
            "n_chars": len(text),
            "n_words": len(text.split()),
            "cyr_share": cyr / max(cyr + lat, 1),
            "n_emails": len(email_re.findall(text)),
            "n_phones": len(phone_re.findall(text)),
            "has_assignment_markers": int(
                "ЗАДАНИЕ" in text.upper() or "Задание" in text
            ),
        }
    )

ru_df = pd.DataFrame(ru_rows).sort_values("n_words")
ru_df.head()

,file,n_chars,n_words,cyr_share,n_emails,n_phones,has_assignment_markers
2,Тетрадь_10.md,4241,310,0.0,0,0,0
20,Тетрадь_28.md,4433,324,0.0,0,0,0
11,Тетрадь_2.md,4426,333,0.0,0,0,0
12,Тетрадь_20.md,4812,346,0.0,0,0,0
16,Тетрадь_24.md,4415,362,0.0,0,0,0


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(ru_df["n_words"], bins=10, ax=axes[0], color="#4C78A8")
axes[0].set_title("Длина русских текстов (слова)")
axes[0].set_xlabel("слов")

sns.boxplot(y=ru_df["n_words"], ax=axes[1], color="#72B7B2")
axes[1].set_title("Разброс длин")
axes[1].set_ylabel("слов")
plt.tight_layout()
fig.savefig(FIG / "ru_lengths.png", dpi=140, bbox_inches="tight")
plt.show()

print(ru_df[["n_words", "n_chars", "cyr_share"]].describe().round(3))
print(
    "\nДоля файлов с кириллицей > 70%:",
    float((ru_df["cyr_share"] > 0.7).mean().round(2)),
)
print("Файлы с email:", int((ru_df["n_emails"] > 0).sum()))
print("Файлы с телефонами:", int((ru_df["n_phones"] > 0).sum()))
print("Файлы с маркером ЗАДАНИЕ:", int(ru_df["has_assignment_markers"].sum()))

       n_words   n_chars  cyr_share
count   27.000    27.000     27.000
mean   438.222  4979.815      0.255
std    104.505  1088.486      0.440
min    310.000  3145.000      0.000
25%    375.500  4429.500      0.000
50%    409.000  4812.000      0.000
75%    463.000  5457.000      0.479
max    742.000  7789.000      0.999

Доля файлов с кириллицей > 70%: 0.26
Файлы с email: 0
Файлы с телефонами: 0
Файлы с маркером ЗАДАНИЕ: 7


/var/folders/dp/1h0qmjhd05qc9y9v11b3h_vm0000gt/T/ipykernel_9283/1211514459.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Вывод**

Русских файлов больше, чем размеченных английских. Текст в основном на русском, email и телефоны простым поиском не нашлись.

Но это не набор «работа + баллы по рубрике». Как примеры, как можно писать фидбэк, файлы полезны. Чтобы считать MAE или F1 на русском, разметки не хватает.


## 4. PDF: получается ли достать текст

Проверяем, читается ли PDF обычным извлечением текста. OCR в первой версии не обещаем.


In [11]:
pdf_rows = []
for pdf_path in pdf_files:
    txt_path = pdf_text_dir / f"{pdf_path.stem}.txt"
    text = (
        txt_path.read_text(encoding="utf-8", errors="replace")
        if txt_path.exists()
        else ""
    )
    letters = sum(ch.isalnum() for ch in text)
    pdf_rows.append(
        {
            "file": pdf_path.name,
            "size_mb": round(pdf_path.stat().st_size / (1024 * 1024), 2),
            "extracted_chars": len(text),
            "alnum_chars": letters,
            "extracted_words": len(text.split()),
            "looks_empty": letters < 200,
            "kind": "metric" if "Метрика" in pdf_path.name else "submission",
        }
    )

pdf_df = pd.DataFrame(pdf_rows).sort_values("extracted_words")
pdf_df

,file,size_mb,extracted_chars,alnum_chars,extracted_words,looks_empty,kind
0,Метрика_done.pdf,0.06,3276,2354,399,False,metric
5,РТ6.pdf,3.65,26526,15498,2742,False,submission
4,РТ30.pdf,4.71,29411,17165,3094,False,submission
2,РТ23.pdf,2.63,29697,17761,3174,False,submission
1,РТ18.pdf,4.05,31597,18847,3298,False,submission
3,РТ25.pdf,4.27,33860,20600,3646,False,submission


In [12]:
plot_pdf = pdf_df[pdf_df["kind"] == "submission"].copy()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=plot_pdf, x="file", y="extracted_words", color="#E45756", ax=ax)
ax.set_title("Слов после pdftotext по PDF работ")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig.savefig(FIG / "pdf_extract_words.png", dpi=140, bbox_inches="tight")
plt.show()

print("Пустых/почти пустых PDF:", int(pdf_df["looks_empty"].sum()))
print(
    pdf_df[["file", "size_mb", "extracted_words", "looks_empty"]].to_string(index=False)
)

Пустых/почти пустых PDF: 0
            file  size_mb  extracted_words  looks_empty
Метрика_done.pdf     0.06              399        False
         РТ6.pdf     3.65             2742        False
        РТ30.pdf     4.71             3094        False
        РТ23.pdf     2.63             3174        False
        РТ18.pdf     4.05             3298        False
        РТ25.pdf     4.27             3646        False


/var/folders/dp/1h0qmjhd05qc9y9v11b3h_vm0000gt/T/ipykernel_9283/1062866384.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Вывод**

Из `РТ*.pdf` текст достаётся, причём довольно много слов. Значит, это не просто картинки-сканы. Сами файлы тяжёлые, внутри, скорее всего, ещё и оформление.

Парсер PDF в MVP нужен. OCR можно не делать сразу, но если текст не извлечётся, надо честно показать ошибку, а не отправлять в модель пустышку.


## 5. Что это даёт для метрик

На `checks_scored` считаем простые статистики по баллам. Саму LLM здесь не гоняем. Какие метрики брать в целом — уже решили в ДЗ1: MAE, Precision, Recall, F1 и то, принимает ли преподаватель черновик.


In [13]:
tmp = []
for i, item in enumerate(checks_scored):
    scores = item.get("criteria_results") or []
    total = sum(int(s) for _, s in scores)
    tmp.append(
        {
            "work_idx": i,
            "task_id": item["task_id"],
            "total_score": total,
            "max_score": len(scores),
        }
    )
work_df = pd.DataFrame(tmp)
work_df["share"] = work_df["total_score"] / work_df["max_score"]
work_df

,work_idx,task_id,total_score,max_score,share
0,0,science_essay_250,7,10,0.7
1,1,science_essay_250,9,10,0.9
2,2,science_essay_250,5,10,0.5
3,3,book_review_150_180,6,10,0.6
4,4,book_review_150_180,9,10,0.9
5,5,book_review_150_180,1,10,0.1
6,6,book_review_150_180,5,10,0.5
7,7,book_review_150_180,10,10,1.0


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(
    work_df["total_score"],
    bins=range(0, int(work_df["max_score"].max()) + 2),
    ax=axes[0],
    color="#B279A2",
)
axes[0].set_title("Суммарный балл работы")
axes[0].set_xlabel("число критериев со score=1")

axes[1].bar(
    ["score=0", "score=1"],
    [(crit_df["score"] == 0).mean(), (crit_df["score"] == 1).mean()],
    color=["#E45756", "#54A24B"],
)
axes[1].set_ylim(0, 1)
axes[1].set_title("Глобальный баланс меток")
axes[1].set_ylabel("доля")
plt.tight_layout()
fig.savefig(FIG / "score_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

print(work_df[["total_score", "share"]].describe().round(3))
print(f"Глобальная доля положительных меток: {crit_df['score'].mean():.3f}")

       total_score  share
count        8.000  8.000
mean         6.500  0.650
std          2.928  0.293
min          1.000  0.100
25%          5.000  0.500
50%          6.500  0.650
75%          9.000  0.900
max         10.000  1.000
Глобальная доля положительных меток: 0.650


/var/folders/dp/1h0qmjhd05qc9y9v11b3h_vm0000gt/T/ipykernel_9283/1286094932.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Вывод по метрикам**

На этих данных логично считать F1 по каждому критерию и MAE по сумме баллов.

Цифры из прошлого года (MAE 1.66 и F1 0.86) относятся к другой, более большой проверке. На наших 8 работах можно повторить способ оценки, но нельзя говорить, что мы уже получили те же 0.86.

Для русского сначала нужна ручная разметка хотя бы части тетрадей или PDF.


## 6. Главные выводы

1. Данных хватает, чтобы разобраться и собрать демо-бейзлайн. Для нормального обучения мало: 2 задания, 8 размеченных английских работ, 27 русских текстов, 5 PDF.

2. Самое ценное — английский набор, где есть задание, критерии, ответ, баллы и фидбэк вместе.

3. По критериям разметка кривовата: часть пунктов почти всегда равна 1, F1 из-за этого может врать.

4. Русских файлов больше, но без баллов. Для русского MVP это главная дыра.

5. PDF сейчас читаются без OCR. Но сканы и пустые файлы всё равно возможны.

6. Почту и телефон простым поиском не нашли. Перед отправкой в модель текст всё равно лучше чистить.

7. Чтобы спокойнее оценивать качество, хотелось бы порядка 50–100 размеченных работ и ещё 20–30 русских с оценками преподавателя.


## 7. Что делать дальше

- Для английского бейзлайна берём `checks_scored` как эталон.
- Русские md и PDF пока держим отдельно: это примеры без разметки.
- При проверке модели смотрим целую работу, а не куски текста из одной работы в train и test сразу.
- Дальше всё равно понадобятся парсинг PDF, очистка личных данных и ответ по рубрике, который преподаватель может поправить.

Этапы после EDA расписаны в `docs/ml_system_design_doc.md`, раздел 2.3.
